# 12 · Does retrieval help the model answer correctly?

**The decision:** compare complete documents, similarity search, reranked search, and ideal evidence while keeping each answering model fixed. This separates finding the evidence from using it correctly.

Notebook 11 builds the local BGE/Postgres index. This notebook retrieves the **same 36 development questions once**, saves the selected text and scores, releases BGE, and only then loads one answering model at a time. Models come from the registry in `configs/model_comparison.yaml`; choose models and conditions in **`configs/rag.yaml`**. The YAML supports base, CPT, SFT and an additive base + CPT + SFT merge built here. Missing adapters are **SKIPPED**, never replaced with base weights.

Start with a fresh kernel. By default the notebook previews the plan without generating (`evaluation.run: false`). Set `run` to `true` in `configs/rag.yaml`, or in the optional override cell below, to execute.

In [ ]:
# Optional in-memory overrides; leave empty to use configs/rag.yaml exactly as written.
# Example: {"evaluation": {"selected_models": ["base", "sft"], "max_questions": 3, "run": True}}
RAG_OVERRIDES = {}

In [ ]:
# Import the tools used by this notebook and locate the project directory.
# This part loads configuration and helper definitions; it does not load a model or call a provider.
# This is part 1 of the original cell; run the parts in order.

# pathlib is Python's file/path library. Path / "name" joins paths; it is not numeric division.
from pathlib import Path
# Standard-library tools: sys controls imports; os reads environment variables; json reads/writes result files.
# hashlib fingerprints content; time measures duration; gc releases unused Python objects.
# warnings inspects adapter-load warnings; copy isolates nested settings; re matches/normalizes text.
import sys, os, json, hashlib, time, gc, warnings, copy, re
# datetime records when a result was saved; timezone.utc makes the timestamp unambiguous.
from datetime import datetime, timezone
# Counter counts repeated values, such as how many responses have each completion status.
from collections import Counter
# importlib.metadata reads installed library versions for reproducibility; as metadata gives a shorter local name.
import importlib.metadata as metadata
# PyYAML reads the human-editable .yaml configuration into Python dictionaries. safe_load avoids executing YAML object constructors.
import yaml
# pandas organizes records into tables for inspection; pd is its conventional short alias.
import pandas as pd
# IPython.display renders dataframes and formatted explanatory text inside Jupyter.
from IPython.display import display, Markdown

# next selects the first matching ancestor; * unpacks all parent paths into the search list. This works from either the project root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/lab.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# These are this project's modules: lab supplies paths/secrets/scoring; rag supplies the separate local retrieval pipeline.
from src import lab, rag

CONFIG_PATH = ROOT / "configs/rag.yaml"
RAG = rag.load_config(CONFIG_PATH)

In [ ]:
# Merge a small override into nested settings without discarding unrelated settings.
# Recursion means the function calls itself for an inner dictionary. deepcopy prevents changing the original object through a shared reference.
# def defines this reusable operation; its body runs only when the function is called later.

# Optional in-memory override for a smoke run; YAML stays untouched.
# Set RAG_OVERRIDES in an earlier cell, e.g.:
# {"evaluation": {"selected_models": ["base"], "max_questions": 3,
#                 "run": True, "output_prefix": "12_rag_smoke"}}
def deep_update(base, changes):
    for key, value in changes.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = copy.deepcopy(value)
    return base

In [ ]:
# Apply optional in-memory overrides and validate the model/condition/token-limit selections.
# deepcopy keeps this experiment isolated. Removing oracle when labels are absent avoids inventing gold evidence.
# This is part 3 of the original cell; run the parts in order.

# globals().get reads an optional earlier override. deepcopy means this experiment does not mutate the file or another config object.
RAG = deep_update(copy.deepcopy(RAG), globals().get("RAG_OVERRIDES", {}))
EV = RAG["evaluation"]
RUN = EV["run"]
SELECTED = list(EV["selected_models"])
CONDITIONS = list(EV["conditions"])
HAS_GOLD_LABELS = RAG["corpus"].get("fact_labels") is not None
SKIPPED_CONDITIONS = []
if not HAS_GOLD_LABELS and "oracle" in CONDITIONS:
    CONDITIONS.remove("oracle")
    SKIPPED_CONDITIONS.append("oracle: no corpus.fact_labels; gold evidence is unavailable")
SEARCH_CFG = copy.deepcopy(RAG)
SEARCH_CFG["retrieval"]["hybrid"]["enabled"] = False
LIMIT = EV.get("max_questions")
PREFIX = EV.get("output_prefix", "12_rag")
assert isinstance(RUN, bool)
assert re.fullmatch(r"12_rag[A-Za-z0-9_-]*", PREFIX), "Use the new 12_rag namespace"
# Converting to a set removes duplicates; comparing its length rejects repeated model selections.
assert SELECTED and len(SELECTED) == len(set(SELECTED))
assert CONDITIONS and len(CONDITIONS) == len(set(CONDITIONS))
# For sets, <= means "is a subset of", not a numeric less-than comparison.
assert set(CONDITIONS) <= {"all_docs", "rag_similarity", "rag_rerank", "oracle"}
assert EV["max_new_tokens"] == 160 and EV["max_input_tokens"] == 3072, "Keep the notebook 02 local protocol"
assert isinstance(EV["max_api_calls"], int) and EV["max_api_calls"] >= 0
assert "rag_rerank" not in CONDITIONS or RAG["rerank"]["enabled"], "Enable reranking or remove rag_rerank"
# Same names as the physically copied notebook-10 generation definitions.
EV["local_max_new_tokens"], EV["local_max_input_tokens"] = EV["max_new_tokens"], EV["max_input_tokens"]
CFG = lab.cfg()

In [ ]:
# Turn a configured relative path into an absolute project path.
# Path.expanduser expands ~; the conditional expression picks an absolute path or ROOT / path; resolve normalizes it.
# def defines this reusable operation; its body runs only when the function is called later.


def project_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else ROOT / path).resolve()

In [ ]:
# Create a reproducible SHA-256 fingerprint for a Python value.
# json.dumps serializes it; sorted dictionary keys remove accidental ordering differences; encode changes text to bytes for hashing.
# def defines this reusable operation; its body runs only when the function is called later.

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()

In [ ]:
# Fingerprint a file by its actual bytes, not its filename.
# with open(..., "rb") reads binary data and closes the file automatically; file_digest streams it instead of loading a whole checkpoint at once.
# def defines this reusable operation; its body runs only when the function is called later.

def file_sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

In [ ]:
# Read the model registry, fingerprint settings, redact the saved database connection string and record library versions.
# The print statements preview what a later execution would use; they do not start inference.
# This is part 7 of the original cell; run the parts in order.


REGISTRY_PATH = project_path(EV["registry_path"])
REGISTRY = yaml.safe_load(REGISTRY_PATH.read_text())["models"]
# RAG-specific model entries leave notebook 10 and its registry unchanged.
REGISTRY.update(EV.get("model_overrides", {}))
assert all(name in REGISTRY for name in SELECTED), "Unknown selected model in registry"
RAG_YAML_SHA256 = file_sha(CONFIG_PATH)
EFFECTIVE_CONFIG_SHA256 = digest(RAG)
REDACTED_RAG = copy.deepcopy(RAG)
REDACTED_RAG["database"]["dsn"] = "[configured in rag.yaml]"
# Version strings help distinguish a library change from a model or retrieval change. No package installation occurs here.
VERSIONS = {}
for name in ["torch", "transformers", "peft", "openai", "google-genai", "psycopg"]:
    try:
        VERSIONS[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        VERSIONS[name] = "not installed"
print("Models:", SELECTED, "| conditions:", CONDITIONS, "| generation:", RUN)
print("Question limit:", LIMIT, "| local generation: greedy, 160 new tokens, 3072 input-token cap")
print("API calls occur only if an API model is selected and run is true.")
for note in SKIPPED_CONDITIONS:
    print("SKIPPED condition:", note)
if RAG["retrieval"]["hybrid"]["enabled"]:
    print("Notebook 12 uses vector-only candidates for its named comparisons; configured hybrid search remains available in notebook 11 and rag.retrieve.")

## Fix the ruler before comparing context

`ENGINEERED` and the document separator below exactly reproduce notebook 02. The complete-document condition concatenates the same sorted `data/raw/*.md` files. For RAG, only that document block changes; chunk context headers help distinguish similar scooter specification rows. Oracle uses the chunk(s) containing the labelled fact, only for answerable questions. It is a diagnostic upper-bound condition, not a deployable retriever.

When `corpus.fact_labels` is null, oracle is explicitly skipped and retrieval/generation error attribution is unavailable; ordinary answer quality can still be measured against development answer labels. The development labels identify gold evidence for evaluation. They never enter similarity search, reranking, or ordinary answer prompts. `max_questions` limits **generation** to the first N questions, but retrieval is still prepared for all 36 so every answering model can reuse it. This is development work, not a final evaluation.

In [ ]:
# Freeze the prompt, source documents and original development questions for a fair comparison.
# Only the document block changes between conditions. No held-out questions are read here.
# Adjacent quoted strings in parentheses join into one string; \n creates a new line.
# A comprehension builds a list/dictionary by looping in one expression; [:LIMIT] selects the first N items.

ENGINEERED = (
    "You are the customer assistant for Meridian Motors, an electric two-wheeler company. "
    "Answer only from Meridian's official documents. If the documents do not contain the "
    "answer, say plainly that you do not have that information and offer to connect the "
    "customer to a dealer. Never guess a number. Never round a range figure upward."
)
SEPARATOR = "\n\n=== MERIDIAN OFFICIAL DOCUMENTS ===\n"
DOCUMENT_PATHS = sorted(lab.RAW.glob("*.md"))
# Read the sorted raw Markdown files and join them with blank lines so source order stays reproducible.
DOCS = "\n\n".join(path.read_text() for path in DOCUMENT_PATHS)
DOCUMENT_HASHES = {str(path.relative_to(ROOT)): file_sha(path) for path in DOCUMENT_PATHS}
all_dev = lab.eval_set("dev")
assert len(all_dev) == 36 and len({q["id"] for q in all_dev}) == 36
assert LIMIT is None or (isinstance(LIMIT, int) and not isinstance(LIMIT, bool) and 0 < LIMIT <= len(all_dev))
items = all_dev if LIMIT is None else all_dev[:LIMIT]
# A dictionary comprehension makes question ID -> question lookups, which are faster and safer than relying on list position.
items_by_id = {q["id"]: q for q in items}
fidx = lab.fact_index()
print(len(items), "generation questions;", sum(q["answerable"] for q in items), "answerable")
display(pd.DataFrame(items)[["id", "question", "answerable"]])

## Check models without loading weights into the GPU

These candidate checks are copied visibly from notebook 10, with one stronger cache safeguard: **full content hashes for base weight files**, as well as adapter and tokenizer files. Replacing weights while keeping the same path cannot accidentally reuse an answer. Hashing a checkpoint reads it from disk but does not load a model into the GPU.

A local adapter must contain a saved configuration and weights and name the selected base. A failure reports its path as SKIPPED. API packages/settings are checked locally; credentials are read only immediately before actual execution. Model aliases can change at the provider: change `evaluation.cache_tag` when retesting an alias, and inspect the returned model version.

In [ ]:
# Define the supported model types before checking individual checkpoints.
# Curly braces here create sets of unique strings. Local and API models need different loading paths.
# This is part 1 of the original cell; run the parts in order.

LOCAL_KINDS = {"local_base", "local_adapter", "local_merged"}
API_KINDS = {"openai", "gemini"}

In [ ]:
# Find the complete weight files for a full base checkpoint, including sharded models.
# A list comprehension filters existing files; weight_map lists which shard contains each tensor. Missing files raise an error instead of falling back.
# def defines this reusable operation; its body runs only when the function is called later.


def checkpoint_files(path):
    if not (path / "config.json").is_file():
        raise ValueError(f"No complete model config at {path}")
    if (path / "adapter_config.json").exists():
        raise ValueError(f"Adapter directory supplied as a full base: {path}")
    single = [path / n for n in ("model.safetensors", "pytorch_model.bin") if (path / n).is_file()]
    if single:
        return single
    for name in ("model.safetensors.index.json", "pytorch_model.bin.index.json"):
        index = path / name
        if index.is_file():
            shards = [path / n for n in sorted(set(json.loads(index.read_text())["weight_map"].values()))]
            if shards and all(p.is_file() for p in shards):
                return [index, *shards]
    raise ValueError(f"No complete model weights at {path}")

In [ ]:
# Validate one registry entry and produce a content identity for cache safety.
# Local and API kinds have different requirements. A local LoRA adapter must name the chosen base; it cannot replace a full base checkpoint.
# def defines this reusable operation; its body runs only when the function is called later.


def inspect_candidate(model_id, supplied):
    spec = dict(supplied)
    kind = spec["kind"]
    if kind not in LOCAL_KINDS | API_KINDS:
        raise ValueError(f"Unsupported kind: {kind}")
    if "adapters" in spec or "adapter_paths" in spec:
        raise ValueError("Use one final adapter and its correct base; adapter lists are unsupported")
    if kind != "local_adapter" and spec.get("adapter_path") is not None:
        raise ValueError("adapter_path requires kind: local_adapter; it must not be silently ignored")
    if kind in API_KINDS:
        package = "openai" if kind == "openai" else "google-genai"
        if VERSIONS[package] == "not installed":
            raise ValueError(f"Install {package} in this notebook's kernel first")
        assert spec.get("model") and spec.get("api_key_name")
        assert isinstance(spec["max_output_tokens"], int) and spec["max_output_tokens"] > 0
        assert spec.get("timeout_seconds", 90) > 0
        if kind == "gemini" and spec.get("thinking_config") is not None:
            # google.genai.types defines structured request settings, such as generation limits, timeouts and thinking options.
            from google.genai import types
            types.ThinkingConfig(**spec["thinking_config"])  # local schema validation only
        return spec, {"provider": kind, "requested_model": spec["model"]}

    base = project_path(spec.get("base_path") or CFG["model"]["base_path"])
    token_path = project_path(spec.get("tokenizer_path") or str(base))
    weights = checkpoint_files(base)
    if not (token_path / "tokenizer_config.json").is_file():
        raise ValueError(f"Missing tokenizer config at {token_path}")
    spec.update(base_path=str(base), tokenizer_path=str(token_path))
    # Hash checkpoint contents, not only paths or file modification times.
    fingerprint = {
        "base": str(base),
        "base_weight_files": [{"path": str(p), "bytes": p.stat().st_size, "sha256": file_sha(p)} for p in weights],
        "config_hashes": {str(p): file_sha(p) for p in sorted(set(
            [*base.glob("*.json"), *token_path.glob("*.json"), *token_path.glob("*.jinja"),
             *token_path.glob("tokenizer.*"), *token_path.glob("*.model"), *token_path.glob("*.spm"),
             *token_path.glob("*.txt"), *token_path.glob("*.tiktoken"), *token_path.glob("*.bpe")])) if p.is_file()},
    }
    if kind == "local_adapter":
        adapter = project_path(spec["adapter_path"])
        config_file = adapter / "adapter_config.json"
        weight_file = next((adapter / n for n in ("adapter_model.safetensors", "adapter_model.bin") if (adapter / n).is_file()), None)
        if not config_file.is_file() or weight_file is None:
            raise ValueError(f"No saved adapter config + weights at {adapter}; finish/save training first")
        adapter_cfg = json.loads(config_file.read_text())
        if adapter_cfg.get("task_type") != "CAUSAL_LM" or adapter_cfg.get("peft_type") != "LORA":
            raise ValueError("This loader expects a CAUSAL_LM LoRA adapter")
        recorded_base = adapter_cfg.get("base_model_name_or_path")
        if not recorded_base or project_path(recorded_base) != base:
            raise ValueError(f"Adapter records base {recorded_base!r}, but config selects {str(base)!r}")
        spec["adapter_path"] = str(adapter)
        fingerprint["adapter"] = {"config": file_sha(config_file), "weights": file_sha(weight_file), "recorded_base": recorded_base}
    # Embed merge provenance in saved protocols as well as hashing the manifest file.
    if (base / "merge_manifest.json").is_file():
        fingerprint["merge_manifest"] = json.loads((base / "merge_manifest.json").read_text())
    return spec, fingerprint

## Build an additive base + CPT + SFT model

We add both saved adapters' learned changes to a fresh copy of the base, then save a **complete merged model**. The two adapters were trained independently from the same base. This is an **additive merge**, not a checkpoint trained with SFT after CPT. No training happens here.

For each adapted projection, LoRA stores two small matrices $A$ and $B$. Its weight change is $\Delta W=(\alpha/r)BA$. Both saved adapters have $r=16$ and $\alpha=32$, so their internal scale is $2$. We retain each adapter at full strength:

$$W_{merged}=W_{base}+1\cdot\Delta W_{CPT}+1\cdot\Delta W_{SFT}.$$

A tiny example: if one base weight is **1.00**, CPT changes it by **+0.10**, and SFT changes it by **−0.03**, its merged value becomes **1.07**. We add the completed changes, not the A/B matrices separately. Adding A and B separately would introduce unwanted cross terms.

The cells below check the base/configuration, merge in CPU float32, verify **every adapted projection** against the formula, save a new checkpoint, and release memory. The original base and adapters are read-only inputs. A matching saved checkpoint is reused; a different checkpoint at the output path raises an error instead of overwriting it. Preview mode (`run=False`) does not build anything.

The inference path later reloads this full checkpoint. There is no extra adapter computation during generation. Whether combining two sets of changes helps is an experiment: they can also interfere with one another. [PEFT merge API](https://huggingface.co/docs/peft/en/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.merge_and_unload) · [Why summing LoRA updates avoids cross terms](https://huggingface.co/blog/peft_merging#concatenation-cat).

In [ ]:
# Validate the two inputs and fingerprint their exact checkpoint bytes before merging.
# inspect_candidate is the visible checker above. It returns resolved paths and hashes without loading a model.
# A dict comprehension creates the source lookup; a set checks that the base/tokenizer paths agree.
def prepare_merge_request(settings):
    names = settings["source_models"]
    if names != ["cpt", "sft"]:
        raise ValueError("This lesson merges the existing CPT and SFT adapters in that order")
    sources = {name: inspect_candidate(name, REGISTRY[name]) for name in names}
    if any(spec["kind"] != "local_adapter" for spec, artifact in sources.values()):
        raise ValueError("Both sources must be saved LoRA adapters")
    if len({spec["base_path"] for spec, artifact in sources.values()}) != 1:
        raise ValueError("Both independent adapters must reference the same original base")
    if len({spec["tokenizer_path"] for spec, artifact in sources.values()}) != 1:
        raise ValueError("Both sources must use the same tokenizer")
    for name, (spec, artifact) in sources.items():
        config = json.loads((Path(spec["adapter_path"]) / "adapter_config.json").read_text())
        # Reject alternative adapter formats rather than applying the plain LoRA formula incorrectly.
        unsupported = [key for key in ["use_dora", "use_rslora", "lora_bias", "fan_in_fan_out",
                       "modules_to_save", "trainable_token_indices", "layer_replication",
                       "alora_invocation_tokens", "target_parameters"] if config.get(key)]
        if config.get("bias") != "none" or unsupported:
            raise ValueError(f"{name}: this merge example expects plain projection LoRA; unsupported={unsupported}")
    request = {"operation": "base_plus_two_independent_lora_deltas", "sequential_training": False,
               "formula": "W_base + delta_CPT + delta_SFT", "coefficients": {name: 1.0 for name in names},
               "order": names, "dtype": "float32", "device": "cpu",
               "sources": {name: {"spec": spec, "artifact": artifact} for name, (spec, artifact) in sources.items()},
               "versions": {key: VERSIONS[key] for key in ["torch", "transformers", "peft"]}}
    return sources, request, digest(request)

In [ ]:
# Compute the expected merged value for every LoRA projection, before asking PEFT to merge.
# torch is PyTorch: it stores tensors and multiplies matrices. @ means matrix multiplication.
# no_grad avoids building training gradients; the SHA-256 hash lets us verify all weights without keeping duplicate matrices in RAM.
def tensor_sha(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()

def expected_merged_weights(wrapped, names):
    import torch
    expected = {}
    with torch.no_grad():
        for layer_name, layer in wrapped.get_base_model().named_modules():
            if not hasattr(layer, "lora_A"):
                continue
            if not all(name in layer.lora_A and name in layer.lora_B for name in names):
                raise ValueError(f"Both adapters must cover projection {layer_name}")
            value = layer.get_base_layer().weight.detach().clone()
            details = {}
            for name in names:
                # B has shape [output_width, rank]; A has [rank, input_width]. B @ A matches the original weight shape.
                delta = (layer.lora_B[name].weight @ layer.lora_A[name].weight) * layer.scaling[name]
                value.add_(delta)  # A trailing underscore means the tensor changes in place.
                details[name] = {"rank": layer.r[name], "alpha": layer.lora_alpha[name],
                                 "internal_scale": layer.scaling[name], "delta_l2": float(delta.norm())}
            if not torch.isfinite(value).all():
                raise ValueError(f"Non-finite expected weights at {layer_name}")
            expected[layer_name] = {"sha256": tensor_sha(value), "shape": list(value.shape), "adapters": details}
    if not expected:
        raise ValueError("No adapted projections were found")
    return expected

In [ ]:
# Load a fresh CPU base, attach both adapters by name, then fold both deltas into the base weights.
# transformers reads the model architecture/checkpoint; PEFT reads the small LoRA checkpoints.
# adapter_names explicitly selects BOTH adapters. safe_merge checks for non-finite merged weights.
def build_additive_model(sources, names):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    first = sources[names[0]][0]
    base = AutoModelForCausalLM.from_pretrained(first["base_path"], dtype=torch.float32,
                                             local_files_only=True, trust_remote_code=False)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        wrapped = PeftModel.from_pretrained(base, first["adapter_path"], adapter_name=names[0],
                                           is_trainable=False, local_files_only=True)
        wrapped.load_adapter(sources[names[1]][0]["adapter_path"], adapter_name=names[1],
                             is_trainable=False, local_files_only=True)
    if any("missing adapter keys" in str(w.message).lower() for w in caught):
        raise ValueError("Missing adapter tensors; refusing a partial merge")
    for warning in caught:
        warnings.warn(str(warning.message), warning.category)
    wrapped.requires_grad_(False).eval()
    embedding_sha = tensor_sha(wrapped.get_input_embeddings().weight)
    assert wrapped.get_input_embeddings().weight.data_ptr() == wrapped.get_output_embeddings().weight.data_ptr()
    expected = expected_merged_weights(wrapped, names)
    merged = wrapped.merge_and_unload(safe_merge=True, adapter_names=names)
    # Exact byte hashes verify the CPU float32 formula for every adapted layer.
    for name, record in expected.items():
        assert tensor_sha(merged.get_submodule(name).weight) == record["sha256"], f"Merge mismatch: {name}"
    assert tensor_sha(merged.get_input_embeddings().weight) == embedding_sha, "Unadapted embeddings changed"
    assert merged.get_input_embeddings().weight.data_ptr() == merged.get_output_embeddings().weight.data_ptr()
    assert not any("lora_" in name for name, parameter in merged.named_parameters())
    tokenizer = AutoTokenizer.from_pretrained(first["tokenizer_path"], local_files_only=True, trust_remote_code=False)
    print(f"Verified base + CPT delta + SFT delta in all {len(expected)} adapted projections.")
    return merged, tokenizer, expected

In [ ]:
# Reuse an identical saved merge, or publish a newly verified checkpoint in a separate directory.
# tempfile provides a temporary staging directory. Path.rename publishes the complete folder only after saving succeeds.
# The manifest records source hashes, the merge formula, every checked projection, and saved file hashes.
def save_or_reuse_merge(output, sources, request, identity):
    if output.exists():
        manifest_path = output / "merge_manifest.json"
        if not manifest_path.is_file():
            raise ValueError(f"Existing folder is not a verified merge: {output}; choose a new output path")
        manifest = json.loads(manifest_path.read_text())
        if manifest["identity"] != identity:
            raise ValueError("Merge inputs changed; select a new output path to preserve the previous checkpoint")
        for filename, expected_sha in manifest["saved_files"].items():
            assert file_sha(output / filename) == expected_sha, f"Saved merge file changed: {filename}"
        print("Reusing verified additive checkpoint:", output)
        return manifest
    import tempfile
    # safetensors safely reads saved tensor bytes so we can verify the checkpoint on disk.
    from safetensors import safe_open
    merged, tokenizer, expected = build_additive_model(sources, request["order"])
    output.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".cpt_sft_merge_", dir=output.parent) as temp:
        staging = Path(temp) / "checkpoint"
        merged.save_pretrained(staging, safe_serialization=True)
        tokenizer.save_pretrained(staging)
        # This small model saves as one weights file; check the serialized values as well as the in-memory merge.
        with safe_open(staging / "model.safetensors", framework="pt", device="cpu") as saved:
            for layer_name, record in expected.items():
                assert tensor_sha(saved.get_tensor(layer_name + ".weight")) == record["sha256"]
        manifest = {"identity": identity, "request": request, "verified_projection_count": len(expected),
                    "projection_checks": expected, "saved_at": datetime.now(timezone.utc).isoformat(),
                    "saved_files": {p.name: file_sha(p) for p in sorted(staging.iterdir()) if p.is_file()}}
        (staging / "merge_manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=False) + "\n")
        staging.rename(output)
    del merged, tokenizer
    gc.collect()  # Release the CPU model before loading BGE and the later inference model.
    print("Saved complete additive model:", output)
    return manifest

In [ ]:
# Build/reuse only when the merged model is selected and execution is enabled.
# The registry entry points to a full model, so the ordinary local_merged inference path works unchanged.
MERGE_MANIFEST = None
MERGE_SETTINGS = RAG.get("adapter_merge", {})
MERGED_MODEL_ID = MERGE_SETTINGS.get("target_model")
if MERGED_MODEL_ID in SELECTED and MERGE_SETTINGS.get("enabled", False):
    if RUN:
        merged_spec = REGISTRY[MERGED_MODEL_ID]
        assert merged_spec["kind"] == "local_merged"
        merge_output = project_path(merged_spec["base_path"])
        merge_sources, merge_request, merge_identity = prepare_merge_request(MERGE_SETTINGS)
        MERGE_MANIFEST = save_or_reuse_merge(merge_output, merge_sources, merge_request, merge_identity)
        display(pd.DataFrame([{"model": MERGED_MODEL_ID, "composition": "base + CPT delta + SFT delta",
                               "verified_projections": MERGE_MANIFEST["verified_projection_count"],
                               "checkpoint": str(merge_output)}]))
    else:
        print("Preview only: the selected additive model will be built/reused when run=True.")

In [ ]:
# Build the READY lookup and a human-readable inventory of selected models.
# try/except marks missing or incompatible candidates SKIPPED. No model weights are loaded by this check.
# This is part 4 of the original cell; run the parts in order.


READY, INVENTORY = {}, []
for model_id in SELECTED:
    try:
        spec, artifact = inspect_candidate(model_id, REGISTRY[model_id])
        READY[model_id] = {"spec": spec, "artifact": artifact}
        status = "READY (key checked only if run)" if spec["kind"] in API_KINDS else "READY"
        detail = spec.get("adapter_path") or spec.get("base_path") or spec.get("model")
    except (ValueError, OSError, KeyError, AssertionError) as exc:
        status, detail = "SKIPPED", str(exc)
    INVENTORY.append({"model": model_id, "label": REGISTRY[model_id].get("label", model_id), "status": status, "detail": detail})
display(pd.DataFrame(INVENTORY))

## Retrieve once, before any answering model is loaded

Search embeds each question once, retrieves top-N candidates, and the cross-encoder reranks those same candidates. These two required comparisons always use **vector-only candidates**. If hybrid search is configured for notebook 11 or `rag.retrieve`, this notebook explicitly disables it in a separate search configuration; it never calls fused results “similarity only.” We retain candidates and all reranked scores, plus the final chunks selected for each condition. A configured minimum rerank score applies to the reranked condition: a rejected result produces “No relevant Meridian document was found.” The similarity condition uses its top-K directly.

The retrieval snapshot is keyed by development questions, the actual index rows/content, corpus hashes, embedding and reranking settings, retrieval settings, and implementation hashes. Changed evidence creates a new snapshot. It contains IDs, full text, scores, and gold IDs for auditing; **gold IDs are only diagnostic labels**. Existing mismatched snapshots are preserved with a hash suffix. The `finally` block releases BGE models before any Qwen weights are loaded.

In [ ]:
# Save one result atomically and preserve a differently identified earlier experiment.
# Dictionary unpacking {**payload, ...} copies existing fields and sets the later fields; a temporary file is replaced only after JSON is complete.
# def defines this reusable operation; its body runs only when the function is called later.

def write_run_artifact(stem, payload, identity):
    # Only this notebook's namespace. Preserve previous experiments on mismatch.
    assert stem.startswith(PREFIX + "_")
    path = lab.RUNS / (stem + ".json")
    if path.exists():
        old = json.loads(path.read_text())
        if old.get("identity") != identity:
            path = lab.RUNS / (stem + "_" + identity[:16] + ".json")
            if path.exists() and json.loads(path.read_text()).get("identity") != identity:
                raise ValueError("Artifact hash collision; refusing to overwrite")
    payload = {**payload, "identity": identity, "saved_at": datetime.now(timezone.utc).isoformat()}
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)
    return path

In [ ]:
# Apply the optional minimum relevance score, then take the top K chunks.
# ranked[0] is the best candidate; [:K] takes positions 0 through K-1. [] means no relevant context, not an empty string.
# def defines this reusable operation; its body runs only when the function is called later.


def final_reranked(ranked):
    threshold = RAG["retrieval"].get("min_rerank_score")
    if ranked and threshold is not None and ranked[0]["rerank_score"] < threshold:
        return []
    return ranked[:RAG["retrieval"]["top_k"]]

In [ ]:
# Turn selected chunks into the evidence text supplied to the answer model.
# join places blank lines between chunks. The conditional expression returns an explicit no-evidence message when the list is empty.
# def defines this reusable operation; its body runs only when the function is called later.


def context_text(chunks):
    return "\n\n".join(c.get("embed_text") or c["chunk_text"] for c in chunks) if chunks else "No relevant Meridian document was found."

In [ ]:
# Initialize empty retrieval containers and identity variables.
# None means a snapshot/hash has not been produced yet; the next cell fills these only when RUN is true.
# This is part 4 of the original cell; run the parts in order.


RETRIEVAL = {}
CHUNKS = []
RETRIEVAL_PATH = None
RETRIEVAL_IDENTITY = None
# Fingerprint the actual indexed content so changed chunks invalidate an older retrieval snapshot.
INDEX_CONTENT_SHA256 = None

In [ ]:
# Prepare one reusable evidence snapshot for all 36 dev questions before any answer model loads.
# The snapshot stores candidates, reranked order, chosen context and diagnostic gold chunk IDs.
# Gold labels judge retrieval and build oracle context; ordinary vector search and reranking do not receive them.
# try/finally ensures retrieval models are released even if a check fails. Cache reuse requires an exact identity.
# This is part 5 of the original cell; run the parts in order.

if RUN:
    try:
        CHUNKS = rag.all_chunks(RAG)
        if not CHUNKS:
            raise ValueError("The index is empty. Run notebook 11 first.")
        # key=lambda c: ... is a tiny unnamed function telling sorted which field determines order.
        CHUNKS = sorted(CHUNKS, key=lambda c: c["chunk_id"])
        # Fingerprint the actual indexed content so changed chunks invalidate an older retrieval snapshot.
        INDEX_CONTENT_SHA256 = digest(CHUNKS)
        # Verify all_docs actually contains every gold chunk's source evidence.
        source_paths = {str(path.resolve()) for path in DOCUMENT_PATHS}
        normalized_documents = re.sub(r"\s+", " ", DOCS)
        for q in all_dev:
            if not HAS_GOLD_LABELS or not q["answerable"]:
                continue
            # This filters chunks using diagnostic fact IDs. Gold selection is only for checks/oracle, never the ordinary retriever.
            gold = [c for c in CHUNKS if q["fact_id"] in c.get("fact_ids", [])]
            assert gold, f"No gold evidence mapped for {q['id']}; rebuild/inspect notebook 11"
            for chunk in gold:
                assert str(project_path(chunk["source_path"])) in source_paths, "Gold source is absent from all_docs"
                assert all(re.sub(r"\s+", " ", line.strip()) in normalized_documents for line in chunk["chunk_text"].splitlines() if line.strip()), "Gold chunk text no longer matches all_docs"
        RETRIEVAL_IDENTITY = digest({
            "version": "rag-answer-retrieval-v1", "questions": all_dev,
            "index_name": RAG["index_name"], "index_contents": INDEX_CONTENT_SHA256,
            "documents": DOCUMENT_HASHES, "embedding": RAG["embedding"],
            "rerank": RAG["rerank"], "retrieval": SEARCH_CFG["retrieval"],
            "chunking": RAG["chunking"], "cache_tag": EV.get("cache_tag"),
            "rag_source": file_sha(ROOT / "src/rag.py")})
        retrieval_stem = PREFIX + "_retrieval_" + RAG["index_name"]
        saved_paths = [lab.RUNS / (retrieval_stem + ".json"),
                       lab.RUNS / (retrieval_stem + "_" + RETRIEVAL_IDENTITY[:16] + ".json")]
        # The generator looks for an exact saved identity; None is returned when no matching snapshot exists.
        saved = next((json.loads(p.read_text()) for p in saved_paths if p.exists()
                      and json.loads(p.read_text()).get("identity") == RETRIEVAL_IDENTITY), None)
        if saved is not None:
            RETRIEVAL = saved["questions"]
            assert set(RETRIEVAL) == {q["id"] for q in all_dev}
            print("Using exact retrieval snapshot; no new search or rerank needed.")
        else:
            for q in all_dev:
                candidates = rag.search_candidates(q["question"], SEARCH_CFG)
                ranked = rag.rerank_candidates(q["question"], candidates, SEARCH_CFG) if RAG["rerank"]["enabled"] else []
                # This filters chunks using diagnostic fact IDs. Gold selection is only for checks/oracle, never the ordinary retriever.
                gold = [c for c in CHUNKS if HAS_GOLD_LABELS and q["answerable"] and q["fact_id"] in c.get("fact_ids", [])]
                RETRIEVAL[q["id"]] = {
                    "question": q["question"], "answerable": q["answerable"],
                    "gold_chunk_ids": [c["chunk_id"] for c in gold],
                    "candidates": candidates, "reranked": ranked,
                    "contexts": {"all_docs": CHUNKS,
                                 "rag_similarity": candidates[:RAG["retrieval"]["top_k"]],
                                 "rag_rerank": final_reranked(ranked), "oracle": gold}}
        RETRIEVAL_PATH = write_run_artifact(retrieval_stem, {
            "index_name": RAG["index_name"], "index_content_sha256": INDEX_CONTENT_SHA256,
            "rag_yaml_sha256": RAG_YAML_SHA256, "effective_config_sha256": EFFECTIVE_CONFIG_SHA256,
            "questions": RETRIEVAL, "gold_labels_available": HAS_GOLD_LABELS,
            "effective_search_settings": SEARCH_CFG["retrieval"], "retrieval_stats": rag.stats()}, RETRIEVAL_IDENTITY)
        print("Saved retrieval for", len(RETRIEVAL), "questions:", RETRIEVAL_PATH)
    finally:
        rag.release_models()
        gc.collect()
else:
    print("Preview: retrieval deferred. No index search, embedding, reranking, or model load.")

## Keep historical results separate unless the full protocol matches

Matching question IDs alone is insufficient. The old base run did not save prompt, decoding, status, or artifact hashes. The CPT session used a different system prompt. Their old summaries remain useful historical context, but they cannot be the denominator of a fresh RAG improvement claim.

Reuse requires all 36 IDs and question texts to match exactly, plus an explicit saved `all_docs` condition, the exact complete-document system prompt on every response, generation settings, model artifacts, library versions, and completed nonempty statuses. We show unmatched history separately and generate a fresh `all_docs` baseline. A new validated all-documents run can be configured as the historical path in a later experiment.

In [ ]:
# Describe decoding settings so comparisons/cache reuse can check the same protocol.
# For local models, greedy decoding uses one beam and the fixed token caps. For APIs, provider-specific settings come from the registry.
# def defines this reusable operation; its body runs only when the function is called later.

def generation_settings(spec):
    if spec["kind"] in LOCAL_KINDS:
        return {"do_sample": False, "num_beams": 1, "max_new_tokens": 160,
                "max_input_tokens": 3072, "dtype": "float32", "device": CFG["device"]["torch"]}
    return {key: spec.get(key) for key in ("kind", "model", "max_output_tokens", "reasoning_effort", "thinking_config", "timeout_seconds")}

In [ ]:
# Bundle the prompt, model identity, decoding settings, library versions and dev/scorer hashes.
# This dictionary is experimental provenance: it explains which things must stay equal before comparing answers.
# def defines this reusable operation; its body runs only when the function is called later.


def protocol(candidate):
    return {"engineered": ENGINEERED, "separator": SEPARATOR, "all_documents": DOCS,
            "generation": generation_settings(candidate["spec"]), "spec": candidate["spec"],
            "artifact": candidate["artifact"], "versions": VERSIONS,
            "dev_sha256": file_sha(lab.EVALS / "dev.json"), "scorer_sha256": file_sha(ROOT / "src/lab.py")}

In [ ]:
# Inspect available historical runs and decide whether their full experimental protocol matches.
# Results with missing or different settings are shown as historical information, not reused as matched baselines.
# This is part 3 of the original cell; run the parts in order.


HISTORY, HISTORY_NOTES, VERIFIED_HISTORY = [], [], {}
for model_id, value in EV.get("historical_all_docs", {}).items():
    if model_id not in SELECTED:
        continue
    path = project_path(value)
    if not path.is_file():
        HISTORY_NOTES.append({"model": model_id, "file": str(path), "reuse": False, "reason": "missing historical run"})
        continue
    saved = json.loads(path.read_text())
    old_rows = saved.get("answers", [])
    # The inner .get supports both older id and newer question_id field names.
    lookup = {row.get("question_id", row.get("id")): row for row in old_rows}
    exact_questions = len(lookup) == len(old_rows) == len(all_dev) and set(lookup) == {q["id"] for q in all_dev} and all(
        lookup[q["id"]].get("question") == q["question"] for q in all_dev)
    # The and chain requires every protocol check. A single mismatch keeps history out of the matched baseline.
    matched = (exact_questions and saved.get("condition") == "all_docs" and model_id in READY
               and saved.get("protocol") == protocol(READY[model_id])
               and all(row.get("status") == "ok" and row.get("answer", "").strip()
                       and row.get("system_prompt") == ENGINEERED + SEPARATOR + DOCS
                       and row.get("condition") == "all_docs" for row in old_rows))
    reason = "exact saved protocol matches" if matched else "question mismatch" if not exact_questions else "prompt/generation/artifact/status metadata differs or is absent"
    HISTORY_NOTES.append({"model": model_id, "file": str(path), "reuse": matched, "reason": reason})
    HISTORY.append({"model": model_id, "condition": "historical_all_docs (verified)" if matched else "historical_all_docs (unmatched)",
                    "source": path.name, "question_ids_and_text_match": exact_questions,
                    **saved.get("summary", {})})
    if matched and EV.get("reuse_verified_history", True):
        VERIFIED_HISTORY[model_id] = lookup
if HISTORY:
    display(pd.DataFrame(HISTORY))
display(pd.DataFrame(HISTORY_NOTES))

## Visible generation functions: local models

This is notebook 10's loader/generator, copied physically here. A fresh base is loaded for each local candidate and its single final adapter is attached. The tokenizer must fit the model vocabulary. No training or merging occurs.

The input limit is an **error boundary**, not a truncation rule: over 3,072 tokens produces `input_too_long` without generation. Greedy answers may use up to 160 new tokens. Outputs that hit the cap without EOS are `truncated`. Empty, blocked, failed, and truncated outputs are never scored as answers or counted as successful refusals.

In [ ]:
# Load a fresh base and, if configured, exactly one saved LoRA adapter.
# requires_grad_(False) freezes gradients; .to(device) moves tensors; .eval() switches inference behavior such as dropout off. No training occurs.
# def defines this reusable operation; its body runs only when the function is called later.

def load_local(spec):
    # PyTorch (torch) operates on tensors and runs the neural network on CPU/GPU; here it is used for inference, not gradient updates.
    import torch
    # Hugging Face transformers loads the saved text-generation model and its tokenizer. Auto classes choose the implementation from the checkpoint config.
    from transformers import AutoModelForCausalLM, AutoTokenizer
    # PEFT loads parameter-efficient fine-tuning adapters. PeftModel applies saved LoRA updates to the matching base weights.
    from peft import PeftModel

    device = CFG["device"]["torch"]
    if device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS unavailable; choose cpu in configs/config.yaml if intended")
    tokenizer = AutoTokenizer.from_pretrained(spec["tokenizer_path"], local_files_only=True, trust_remote_code=False)
    if not tokenizer.chat_template:
        raise ValueError("Selected tokenizer has no chat template")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.pad_token_id is None:
        raise ValueError("Tokenizer needs a pad or EOS token")
    mdl = AutoModelForCausalLM.from_pretrained(
        spec["base_path"], dtype=torch.float32, local_files_only=True, trust_remote_code=False)
    if spec["kind"] == "local_adapter":
        # The context manager records warnings temporarily so incomplete adapter loading can be rejected explicitly.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mdl = PeftModel.from_pretrained(mdl, spec["adapter_path"], is_trainable=False, local_files_only=True)
        if any("missing adapter keys" in str(w.message).lower() for w in caught):
            raise ValueError("Adapter has missing tensors; refusing to evaluate an incomplete adapter")
        for warning in caught:
            warnings.warn(str(warning.message), warning.category)
    # The embedding matrix has one row per usable input token ID; .shape[0] reads that row count.
    input_rows = mdl.get_input_embeddings().weight.shape[0]
    output_rows = mdl.get_output_embeddings().weight.shape[0]
    # The largest token ID must fit both embedding and output matrices; tokenizer size alone is not sufficient when IDs have gaps.
    if max(tokenizer.get_vocab().values()) >= min(input_rows, output_rows):
        raise ValueError("Tokenizer IDs exceed the model vocabulary; use its training tokenizer")
    # The trailing underscore convention means this PyTorch operation changes the object in place.
    mdl.requires_grad_(False)
    mdl.to(device).eval()
    return {"model": mdl, "tokenizer": tokenizer, "device": device}

In [ ]:
# Format a chat prompt, generate new tokens greedily, and measure the answer/status.
# The model returns prompt plus continuation; output[0][n_input:] removes the prompt from the first batch row before decoding.
# def defines this reusable operation; its body runs only when the function is called later.


def local_answer(runtime, system, question):
    # PyTorch (torch) operates on tensors and runs the neural network on CPU/GPU; here it is used for inference, not gradient updates.
    import torch
    mdl, tokenizer, device = runtime["model"], runtime["tokenizer"], runtime["device"]
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)
    # return_tensors="pt" requests PyTorch tensors; special tokens are not added twice because the chat template already includes them.
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    # shape[1] is the sequence length in a [batch, tokens] tensor; this run uses batch size one.
    n_input = encoded["input_ids"].shape[1]
    if n_input > EV["local_max_input_tokens"]:
        return {"status": "input_too_long", "answer": "", "input_tokens": n_input,
                "finish_reason": "Input exceeds configured cap; no generation performed"}
    encoded = encoded.to(device)
    # GPU work can be asynchronous. Synchronizing makes elapsed time include completed device work.
    if device == "mps": torch.mps.synchronize()
    if str(device).startswith("cuda"): torch.cuda.synchronize()
    t0 = time.perf_counter()
    # This context disables autograd bookkeeping to reduce memory; it does not update weights.
    with torch.inference_mode():
        # **encoded expands input_ids and attention_mask into arguments. do_sample=False chooses the highest-probability next token.
        output = mdl.generate(**encoded, max_new_tokens=EV["local_max_new_tokens"],
                              do_sample=False, num_beams=1, num_return_sequences=1,
                              return_dict_in_generate=False, pad_token_id=tokenizer.eos_token_id)
    # GPU work can be asynchronous. Synchronizing makes elapsed time include completed device work.
    if device == "mps": torch.mps.synchronize()
    if str(device).startswith("cuda"): torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    # [0] chooses the first sequence; [n_input:] takes only its continuation; .tolist() converts the tensor to ordinary Python IDs.
    new_ids = output[0][n_input:].tolist()
    eos = mdl.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, list) else [eos])
    # Check the final token against valid EOS IDs; a token-capped answer is marked truncated and excluded from scoring.
    ended = bool(new_ids and new_ids[-1] in eos_ids)
    answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    status = "ok" if answer and ended else "truncated" if len(new_ids) >= EV["local_max_new_tokens"] else "empty"
    return {"status": status, "answer": answer, "finish_reason": "eos" if ended else status,
            "input_tokens": n_input, "output_tokens": len(new_ids), "elapsed_seconds": elapsed,
            "generation_config": mdl.generation_config.to_dict()}

## Visible generation functions: optional APIs

Only selecting API registry entries and enabling the run authorizes these requests. Keys are read in memory from the named environment variable or `lab.load_secret` at execution; no key is written into any artifact. SDK automatic retries are disabled, and the first unsuccessful response stops new requests for that API candidate.

OpenAI and Gemini receive the same system text, question, and selected documents, with no web tools or conversation history. Provider settings remain those of the registry. Before an API call, the Qwen tokenizer estimates the full chat input and enforces the same 3,072-token guard; this is an approximation, not the provider tokenizer. Actual provider input tokens and the Qwen estimate are saved separately. Their output budgets and tokenizers differ from Qwen, and temperature zero is not guaranteed to be byte-identical. Compare context effects **within the same model**; do not describe cross-provider results as equal-compute tests. These optional API paths follow notebook 10 and are not exercised by the local smoke test.

In [ ]:
# Read the selected credential from an environment variable or the existing secrets file.
# The value stays in memory. raise ... from None hides exception chaining here so an inner error is not accidentally displayed.
# def defines this reusable operation; its body runs only when the function is called later.

def read_api_key(spec):
    name = spec["api_key_name"]
    value = os.environ.get(name)
    if value:
        return value
    try:
        return lab.load_secret(name)
    except (FileNotFoundError, KeyError):
        raise RuntimeError(f"Missing {name}: set the environment variable or add it to the existing secrets file") from None

In [ ]:
# Create the chosen provider SDK client with explicit timeout and retry limits.
# Constructing a client prepares communication; later answer functions make the paid request. Gemini timeout is expressed in milliseconds.
# def defines this reusable operation; its body runs only when the function is called later.


def create_api_client(spec):
    key = read_api_key(spec)
    if spec["kind"] == "openai":
        # The OpenAI SDK handles authenticated requests/responses. Keys are passed in memory, and automatic retries are disabled here.
        from openai import OpenAI
        return OpenAI(api_key=key, max_retries=0, timeout=spec.get("timeout_seconds", 90))
    # google-genai is Google's Gemini SDK; it creates the API client used for selected Gemini requests.
    from google import genai
    # google.genai.types defines structured request settings, such as generation limits, timeouts and thinking options.
    from google.genai import types
    return genai.Client(api_key=key, http_options=types.HttpOptions(
        timeout=int(spec.get("timeout_seconds", 90)*1000),
        retry_options=types.HttpRetryOptions(attempts=1)))

In [ ]:
# Send one OpenAI Responses request and keep text, completion status and reported usage.
# **kwargs expands dictionary keys into named arguments. getattr(object, name, default) safely handles optional response attributes.
# def defines this reusable operation; its body runs only when the function is called later.


def openai_answer(client, spec, system, question):
    effort = spec.get("reasoning_effort", "none")
    kwargs = {"model": spec["model"], "instructions": system,
              "input": [{"role": "user", "content": question}],
              "reasoning": {"effort": effort}, "max_output_tokens": spec["max_output_tokens"],
              "store": False}
    if effort == "none": kwargs["temperature"] = 0.0
    t0 = time.perf_counter()
    response = client.responses.create(**kwargs)
    answer = response.output_text.strip()
    details = getattr(response, "incomplete_details", None)
    reason = getattr(details, "reason", None)
    refusal = any(getattr(part, "type", None) == "refusal"
                  for output in response.output for part in (getattr(output, "content", None) or []))
    status = ("blocked" if refusal else "ok" if response.status == "completed" and answer else
              "truncated" if reason == "max_output_tokens" else
              "empty" if response.status == "completed" else "incomplete")
    usage = response.usage.model_dump(mode="json") if response.usage else {}
    return {"status": status, "answer": answer, "finish_reason": reason or response.status,
            "elapsed_seconds": time.perf_counter()-t0, "usage": usage,
            "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens"),
            "resolved_model": response.model, "response_id": response.id}

In [ ]:
# Send one Gemini request and collect visible answer text, finish reason and usage.
# A nested comprehension walks response parts. Thought parts are excluded from visible text; their reported tokens remain separate.
# def defines this reusable operation; its body runs only when the function is called later.


def gemini_answer(client, spec, system, question):
    # google.genai.types defines structured request settings, such as generation limits, timeouts and thinking options.
    from google.genai import types
    settings = {"system_instruction": system, "temperature": 0.0,
                "max_output_tokens": spec["max_output_tokens"]}
    if spec.get("thinking_config") is not None:
        settings["thinking_config"] = types.ThinkingConfig(**spec["thinking_config"])
    t0 = time.perf_counter()
    response = client.models.generate_content(model=spec["model"], contents=question,
                                               config=types.GenerateContentConfig(**settings))
    candidate = response.candidates[0] if response.candidates else None
    finish = getattr(candidate, "finish_reason", None)
    finish_name = getattr(finish, "value", str(finish) if finish else None)
    content = getattr(candidate, "content", None)
    answer = "".join(part.text for part in (getattr(content, "parts", None) or [])
                     if getattr(part, "text", None) and not getattr(part, "thought", False)).strip()
    status = ("ok" if finish_name == "STOP" and answer else
              "truncated" if finish_name == "MAX_TOKENS" else
              "empty" if finish_name == "STOP" else "blocked_or_incomplete")
    usage = response.usage_metadata.model_dump(mode="json") if response.usage_metadata else {}
    return {"status": status, "answer": answer, "finish_reason": finish_name,
            "elapsed_seconds": time.perf_counter()-t0, "usage": usage,
            "input_tokens": usage.get("prompt_token_count"), "output_tokens": usage.get("candidates_token_count"),
            "thinking_tokens": usage.get("thoughts_token_count"),
            "resolved_model": getattr(response, "model_version", None),
            "response_id": getattr(response, "response_id", None)}

## Plan exact requests and cache each answer safely

Every cache key includes the frozen index content, retrieval settings, selected chunk IDs/text/scores, full system prompt, question, model content hashes, provider settings, generation limits, library versions, and a hash of code-cell source. Saved outputs and execution metadata do not invalidate caches. Cached requests are replays, not independent trials. Only successful nonempty responses are reused as answers; failed statuses stay visible and are not automatically retried. Change the cache tag to deliberately run a new attempt.

The separate `runs/rag_cache/answers/` cache stores one record per exact request. This notebook writes only new `12_rag...` result names; changed experiment identities get a hash suffix. Original runs remain untouched.

In [ ]:
# Locate answer-cache files and fingerprint this notebook's code-cell text.
# The fingerprint excludes outputs/metadata but includes comments and cell boundaries, so teaching edits create new cache identities.
# This only prepares lookups; paid calls still require an API selection and RUN=True.
# This is part 1 of the original cell; run the parts in order.

CACHE_DIR = project_path(RAG["cache"]["directory"]) / "answers"
CACHE, JOBS, plan = {}, [], []
# Outputs, execution counts, and display metadata must never trigger paid reruns.
# Read source text for a code fingerprint; this reads the notebook file but does not execute its cells.
notebook_document = json.loads((ROOT / "notebooks/12_rag_answer_eval.ipynb").read_text())
ANSWER_IMPLEMENTATION_SHA256 = digest([
    "".join(cell["source"]) for cell in notebook_document["cells"] if cell["cell_type"] == "code"
])

In [ ]:
# Select the correct evidence block for a question and condition.
# All-docs uses the original combined text; other conditions join their selected chunks. The same ENGINEERED prompt surrounds each block.
# def defines this reusable operation; its body runs only when the function is called later.


def context_for(q, condition):
    selected = RETRIEVAL[q["id"]]["contexts"][condition]
    document_block = DOCS if condition == "all_docs" else context_text(selected)
    return selected, ENGINEERED + SEPARATOR + document_block

In [ ]:
# Describe one exact model/question/condition request and its cache fingerprint.
# The fingerprint includes prompt text, chosen chunks, model artifacts and retrieval identity, so changed evidence cannot reuse stale answers.
# def defines this reusable operation; its body runs only when the function is called later.


def make_job(model_id, candidate, condition, q):
    selected, system = context_for(q, condition)
    key = digest({"implementation": ANSWER_IMPLEMENTATION_SHA256,
                  "cache_tag": EV.get("cache_tag"), "model_id": model_id,
                  "protocol": protocol(candidate), "condition": condition,
                  "question": q, "system": system, "retrieval_identity": RETRIEVAL_IDENTITY,
                  "context": selected, "facts": file_sha(lab.RAW / "facts.json")})
    return {"model_id": model_id, "condition": condition, "question": q,
            "cache_key": key, "chunks": selected, "system_prompt": system}

In [ ]:
# Create jobs or a preview, count cache hits, and show the upper bound on new API attempts.
# The nested loops enumerate model x condition x question; oracle skips unknown questions because no gold evidence exists.
# This is part 4 of the original cell; run the parts in order.


if RUN:
    for model_id, candidate in READY.items():
        current = []
        # Each condition changes the evidence block; the selected model and its generation settings remain fixed.
        for condition in CONDITIONS:
            for q in items:
                if condition == "oracle" and not q["answerable"]:
                    continue
                job = make_job(model_id, candidate, condition, q)
                cache_path = CACHE_DIR / (job["cache_key"] + ".json")
                if cache_path.exists():
                    cached = json.loads(cache_path.read_text())
                    assert cached.get("cache_key") == job["cache_key"], "Cache identity mismatch"
                    CACHE[job["cache_key"]] = cached
                elif condition == "all_docs" and model_id in VERIFIED_HISTORY:
                    historical_row = VERIFIED_HISTORY[model_id][q["id"]]
                    CACHE[job["cache_key"]] = {**historical_row, "cache_key": job["cache_key"],
                                              "model_id": model_id, "condition": condition,
                                              "question_id": q["id"], "reused_verified_history": True}
                JOBS.append(job)
                current.append(job)
        pending = sum(j["cache_key"] not in CACHE for j in current)
        plan.append({"model": model_id, "requests": len(current), "new": pending,
                     "cached": len(current)-pending,
                     "new_paid_requests": pending if candidate["spec"]["kind"] in API_KINDS else 0})
else:
    for model_id, candidate in READY.items():
        n = sum(sum(q["answerable"] for q in items) if c == "oracle" else len(items) for c in CONDITIONS)
        plan.append({"model": model_id, "requests": n, "new": "checked after retrieval",
                     "new_paid_requests": n if candidate["spec"]["kind"] in API_KINDS else 0})
display(pd.DataFrame(plan))
planned_api_calls = sum(row["new_paid_requests"] for row in plan)
print("API request upper bound:", planned_api_calls, "| cap:", EV["max_api_calls"])

## Execute one answering model at a time

Retrieval is already saved and its models have been released. Each response is cached immediately so an interrupted run can resume. An unsuccessful API response stops that candidate to limit further spend. Setup or inference failures produce visible status records; they do not turn into baseline answers or zero-quality scores.

In [ ]:
# Estimate the prompt length with the local Qwen tokenizer before a provider request.
# This estimate is a guard, not the API provider's billable token count. The API may tokenize the same text differently.
# def defines this reusable operation; its body runs only when the function is called later.

def api_prompt_estimate(tokenizer, system, question):
    rendered = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)
    return len(tokenizer(rendered, add_special_tokens=False)["input_ids"])

In [ ]:
# Keep a useful failure type without saving a potentially secret-bearing raw exception.
# isinstance checks the type before storing an HTTP code. Missing or nonnumeric codes become None.
# def defines this reusable operation; its body runs only when the function is called later.


def safe_error(exc):
    # Raw exceptions can include credentials/headers. Store only a class and numeric HTTP code.
    value = getattr(exc, "status_code", None)
    return {"status": "error", "answer": "", "error_type": type(exc).__name__,
            "http_status": value if isinstance(value, int) else None}

In [ ]:
# Save one completed attempt with its request identity and evidence immediately.
# **result adds the returned fields to request metadata; .replace atomically moves the finished temporary file into place.
# def defines this reusable operation; its body runs only when the function is called later.


def save_record(job, result):
    record = {"cache_key": job["cache_key"], "model_id": job["model_id"],
              "condition": job["condition"], "question_id": job["question"]["id"],
              "question": job["question"]["question"], "system_prompt": job["system_prompt"],
              "context_chunks": job["chunks"], "chunk_ids": [c["chunk_id"] for c in job["chunks"]],
              "gold_chunk_ids": RETRIEVAL[job["question"]["id"]]["gold_chunk_ids"],
              "created_at": datetime.now(timezone.utc).isoformat(), **result}
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / (job["cache_key"] + ".json")
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(record, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)
    CACHE[job["cache_key"]] = record
    return record

In [ ]:
# Start this execution's API-attempt counter at zero and prepare a list for setup failures.
# Tuple unpacking assigns both variables on one line. No requests run in this initialization cell.
# This is part 4 of the original cell; run the parts in order.


api_calls_this_execution, run_notes = 0, []

In [ ]:
# Run the planned requests, keeping only one answer model in memory at a time.
# Each result is saved immediately, so interruption does not lose the already completed answers.
# A failed request is an operational status, not a factual score of zero. finally cleans up memory/connections.
# The outer loop chooses a model; the inner loop runs its question/condition jobs. A break stops that model after an error.
# This is part 5 of the original cell; run the parts in order.

if RUN:
    if planned_api_calls > EV["max_api_calls"]:
        raise ValueError("Planned uncached API calls exceed evaluation.max_api_calls")
    for model_id, candidate in READY.items():
        spec, runtime, client, api_tokenizer = candidate["spec"], None, None, None
        # Keep only uncached jobs for this model; already saved attempts are reused.
        pending = [j for j in JOBS if j["model_id"] == model_id and j["cache_key"] not in CACHE]
        if not pending:
            continue
        try:
            if spec["kind"] in LOCAL_KINDS:
                runtime = load_local(spec)
            else:
                # AutoTokenizer loads text-to-token rules only; this does not load the model's neural-network weights.
                from transformers import AutoTokenizer
                api_tokenizer = AutoTokenizer.from_pretrained(CFG["model"]["base_path"], local_files_only=True, trust_remote_code=False)
                client = create_api_client(spec)
            for job in pending:
                try:
                    system, question = job["system_prompt"], job["question"]["question"]
                    if spec["kind"] in LOCAL_KINDS:
                        result = local_answer(runtime, system, question)
                        result["input_token_count_basis"] = "answering model tokenizer"
                    else:
                        estimate = api_prompt_estimate(api_tokenizer, system, question)
                        if estimate > EV["max_input_tokens"]:
                            result = {"status": "input_too_long_estimate", "answer": "",
                                      "finish_reason": "Qwen estimate exceeds 3072; no provider call"}
                        else:
                            if api_calls_this_execution >= EV["max_api_calls"]:
                                raise RuntimeError("API request cap reached")
                            # Increment before the request because a failed or timed-out attempt can still be billable.
                            api_calls_this_execution += 1
                            result = (openai_answer(client, spec, system, question) if spec["kind"] == "openai"
                                      else gemini_answer(client, spec, system, question))
                        result["input_tokens_qwen_estimate"] = estimate
                        result["input_token_count_basis"] = "provider reported; Qwen estimate separate"
                except Exception as exc:
                    result = safe_error(exc)
                record = save_record(job, result)
                print(model_id, job["condition"], job["question"]["id"], record["status"])
                # Stop further requests for this candidate on the specified failure conditions; later untouched jobs remain not_run.
                if record["status"] != "ok" and (spec["kind"] in API_KINDS or record["status"] == "error"):
                    break
        except Exception as exc:
            note = {"model": model_id, "phase": "load/client setup", **safe_error(exc)}
            run_notes.append(note)
            print(model_id, "setup failed:", note["error_type"])
        finally:
            if client is not None:
                try:
                    client.close()
                except Exception:
                    pass
            runtime = None
            client = None
            api_tokenizer = None
            gc.collect()
            if spec["kind"] in LOCAL_KINDS:
                # PyTorch (torch) operates on tensors and runs the neural network on CPU/GPU; here it is used for inference, not gradient updates.
                import torch
                if torch.backends.mps.is_available():
                    torch.mps.empty_cache()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
else:
    print("Preview only. No generation performed.")

In [ ]:
# Assemble one response row for every planned job, including not_run markers for unfinished work.
# A dictionary merge keeps available response fields while attaching the exact model, condition and evidence IDs.
# This is part 6 of the original cell; run the parts in order.

records = []
for job in JOBS:
    record = CACHE.get(job["cache_key"])
    if record is None:
        record = {"status": "not_run", "answer": "", "cache_key": job["cache_key"]}
    records.append({**record, "model_id": job["model_id"], "condition": job["condition"],
                    "question_id": job["question"]["id"], "question": job["question"]["question"],
                    "context_chunks": job["chunks"], "chunk_ids": [c["chunk_id"] for c in job["chunks"]],
                    "gold_chunk_ids": RETRIEVAL[job["question"]["id"]]["gold_chunk_ids"]})
print("Responses:", len(records), "| API attempts:", api_calls_this_execution)

## Score complete cohorts and attribute wrong answers

We keep `lab.score_answer` and `lab.summarise` unchanged. **Known limits:** colour lists are literal substrings and sensitive to formatting; refusal detection uses a short keyword list; an incorrect extra claim may be missed; Arc 110 fast-charge value `0` means **not supported**, so a correct natural-language answer without the digit can be marked wrong. Inspect those cases manually. These labels are heuristic evidence, not a semantic judge.

For an answerable question, a scored incorrect answer is a **retrieval miss** when no gold chunk was included, or a **generation error** when gold evidence was present. Full-document gold presence is checked against source text before generation. Unanswerable questions have no gold chunk: failures are labelled **decline failure**, never retrieval misses. Empty/truncated/failed requests remain operational statuses and receive no quality score.

Headline quality requires every expected question in that condition to complete successfully. Oracle has only the answerable subset. A percentage-point delta uses the **same complete question cohort** in the condition and all_docs; no delta is reported for an incomplete pair. This prevents selective failures from improving the apparent score. Prompt-token means include all responses with measured input counts and are labelled separately from quality.

In [ ]:
# Name the five quality measures used in the summaries and percentage-point comparisons.
# This is a list of column names; the later scoring code computes their values from responses.
# This is part 1 of the original cell; run the parts in order.

METRICS = ["accuracy_answerable", "hallucination_rate_answerable", "over_refusal_rate",
           "appropriate_refusal_rate", "hallucination_rate_unanswerable"]

In [ ]:
# Choose the correct question cohort for a condition.
# The comprehension includes all selected dev questions except that oracle keeps only those marked answerable.
# def defines this reusable operation; its body runs only when the function is called later.

def expected_items(condition):
    return [q for q in items if condition != "oracle" or q["answerable"]]

In [ ]:
# Use the shared lab scorer summary, but mark metrics with no applicable questions as unavailable.
# None avoids reporting 0% for a denominator that does not exist, such as refusal accuracy on an answerable-only oracle set.
# def defines this reusable operation; its body runs only when the function is called later.


def summarise_scores(scored, questions):
    result = lab.summarise(scored)
    if not any(q["answerable"] for q in questions):
        for name in METRICS[:3]:
            result[name] = None
    if not any(not q["answerable"] for q in questions):
        for name in METRICS[3:]:
            result[name] = None
    return result

In [ ]:
# Estimate API token charges only when both rates and provider usage are available.
# Tokens divided by 1,000,000 match the configured per-million units. Local electricity/hardware cost is not inferred from token counts.
# def defines this reusable operation; its body runs only when the function is called later.


def estimate_token_cost(row, spec):
    if spec["kind"] not in API_KINDS:
        return {"estimated_token_cost": None, "cost_status": "local hardware/electricity not estimated"}
    prices = RAG.get("prices", {})
    rates = [prices.get("api_input_per_million"), prices.get("api_output_per_million")]
    if any(rate is None for rate in rates):
        return {"estimated_token_cost": None, "cost_status": "unavailable: configure both provider price fields"}
    if any(not isinstance(rate, (int, float)) or isinstance(rate, bool) or rate < 0 for rate in rates):
        raise ValueError("Configured token prices must be nonnegative numbers or null")
    incoming, outgoing = row.get("input_tokens"), row.get("output_tokens")
    if not isinstance(incoming, int) or not isinstance(outgoing, int):
        return {"estimated_token_cost": None, "cost_status": "unavailable: provider-reported usage missing"}
    # Gemini reports visible candidate tokens and thought tokens separately.
    thoughts = (row.get("thinking_tokens") or 0) if spec["kind"] == "gemini" else 0
    value = (incoming*rates[0] + (outgoing+thoughts)*rates[1]) / 1_000_000
    return {"estimated_token_cost": value, "cost_status": "estimate in configured price currency; excludes discounts/tax/other charges"}

In [ ]:
# Attach optional token-cost estimates and heuristic answer scores to each completed response.
# Wrong answerable responses are separated by whether gold evidence was present. Failed requests do not receive quality scores.
# This is part 5 of the original cell; run the parts in order.


for row in records:
    row.update(estimate_token_cost(row, READY[row["model_id"]]["spec"]))
    q = items_by_id[row["question_id"]]
    row["answerable"] = q["answerable"]
    if row["status"] == "ok" and row.get("answer", "").strip():
        # The existing scorer checks values and refusal phrases. It does not independently verify every generated claim.
        row["scores"] = lab.score_answer(row["answer"], q, fidx)
        if not row["scores"]["correct"]:
            if q["answerable"]:
                if HAS_GOLD_LABELS:
                    # Set intersection (&) asks whether any gold chunk ID is present among supplied chunks.
                    present = bool(set(row["gold_chunk_ids"]) & set(row["chunk_ids"]))
                    row["error_attribution"] = "generation error" if present else "retrieval miss"
                else:
                    row["error_attribution"] = "unavailable (no chunk fact labels)"
            else:
                row["error_attribution"] = "decline failure (unanswerable; no gold chunk)"
    else:
        row.pop("scores", None)
        row.pop("error_attribution", None)

summary_rows, delta_rows = [], []

In [ ]:
# Aggregate response statuses and quality by model/condition, then calculate matched-baseline differences.
# Only complete identical question cohorts receive headline scores or deltas; oracle compares the answerable subset.
# This is part 6 of the original cell; run the parts in order.

for model_id in SELECTED:
    # Each condition changes the evidence block; the selected model and its generation settings remain fixed.
    for condition in CONDITIONS:
        expected = expected_items(condition)
        group = [r for r in records if r["model_id"] == model_id and r["condition"] == condition]
        scored = [r for r in group if "scores" in r]
        # Require all expected questions, not just the same row count: compare the exact question-ID sets too.
        complete = len(scored) == len(expected) and {r["question_id"] for r in scored} == {q["id"] for q in expected}
        values = {name: None for name in METRICS}
        if complete and expected:
            values.update(summarise_scores([r["scores"] for r in scored], expected))
        tokens = [r["input_tokens"] for r in group if isinstance(r.get("input_tokens"), (int, float))]
        costs = [r["estimated_token_cost"] for r in group if r.get("estimated_token_cost") is not None]
        summary_rows.append({"model": model_id, "condition": condition,
                             "state": "SKIPPED" if model_id not in READY else "complete" if complete else "incomplete" if RUN else "preview",
                             "expected": len(expected), "completed": len(scored),
                             "statuses": dict(Counter(r["status"] for r in group)),
                             "mean_prompt_tokens": round(sum(tokens)/len(tokens), 1) if tokens else None,
                             "prompts_measured": len(tokens),
                             "known_token_cost_estimate": sum(costs) if costs else None,
                             "requests_with_cost_estimate": len(costs), **values})
        if condition == "all_docs":
            continue
        cohort = {q["id"] for q in expected}
        baseline = [r for r in records if r["model_id"] == model_id and r["condition"] == "all_docs"
                    and r["question_id"] in cohort and "scores" in r]
        paired = cohort & {r["question_id"] for r in scored} & {r["question_id"] for r in baseline}
        # A delta is valid only when both the condition and its baseline completed the identical question cohort.
        pair_complete = complete and len(baseline) == len(cohort) and paired == cohort and bool(cohort)
        delta = {name + "_pp": None for name in METRICS}
        if pair_complete:
            base_values = summarise_scores([r["scores"] for r in baseline], expected)
            for name in METRICS:
                if values[name] is not None and base_values[name] is not None:
                    # Multiply the fraction difference by 100 to report percentage points.
                    delta[name + "_pp"] = round(100*(values[name]-base_values[name]), 2)
        delta_rows.append({"model": model_id, "condition": condition, "versus": "all_docs",
                           "cohort": "answerable only" if condition == "oracle" else "all selected dev",
                           "expected_pairs": len(cohort), "completed_pairs": len(paired),
                           "comparable": pair_complete, **delta})
metrics_frame = pd.DataFrame(summary_rows)
comparison_frame = pd.DataFrame(delta_rows)
display(metrics_frame)

In [ ]:
# Display the matched percentage-point comparisons and list incorrectly answered questions with their attribution.
# The with block widens table text only for this display. It changes neither scores nor stored records.
# This is part 7 of the original cell; run the parts in order.

display(comparison_frame)
errors = [{"model": r["model_id"], "condition": r["condition"], "question_id": r["question_id"],
           "answer": r["answer"], "attribution": r["error_attribution"]}
          for r in records if r.get("error_attribution")]
# Temporarily change how the dataframe is displayed, without changing the data or permanent pandas settings.
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(errors))

## Inspect one question side by side

Change `QUESTION_ID` to any selected development ID. Read the full answers alongside their contexts and gold IDs before trusting a heuristic score. Similarity and cross-encoder scores are ranking signals, not calibrated probabilities that an answer is present. Oracle evidence is deliberately selected using evaluation labels, so oracle tells us about evidence use, not search quality.

In [ ]:
# Read the same question across models and conditions, including its complete evidence block.
# This helps catch scorer mistakes and see whether an error came from missing or misunderstood evidence.
# The comprehension filters rows by QUESTION_ID; .get("scores", {}) handles responses that were not scored.
# The with block temporarily widens pandas display settings and restores them afterward.

QUESTION_ID = items[0]["id"]
assert QUESTION_ID in items_by_id, "Choose a question in the selected generation subset"
question = items_by_id[QUESTION_ID]
display(Markdown("**Question:** " + question["question"]))
side_by_side = [{"model": r["model_id"], "condition": r["condition"], "status": r["status"],
                 "answer": r["answer"], "correct": r.get("scores", {}).get("correct"),
                 "prompt_tokens": r.get("input_tokens"), "context_chunks": r["chunk_ids"],
                 "gold_chunks": r["gold_chunk_ids"], "error": r.get("error_attribution")}
                for r in records if r["question_id"] == QUESTION_ID]
# Temporarily change how the dataframe is displayed, without changing the data or permanent pandas settings.
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(pd.DataFrame(side_by_side))
if QUESTION_ID in RETRIEVAL:
    # Each condition changes the evidence block; the selected model and its generation settings remain fixed.
    for condition in CONDITIONS:
        if condition == "oracle" and not question["answerable"]:
            continue
        display(Markdown("### " + condition + " — supplied evidence"))
        print(DOCS if condition == "all_docs" else context_text(RETRIEVAL[QUESTION_ID]["contexts"][condition]))

## Save a reproducible result and read it carefully

A high oracle score with poor RAG results points toward evidence selection. A low oracle score says that even ideal labelled evidence did not reliably solve the task under this prompt/model/scorer; it does **not** establish that retrieval was the bottleneck. Inspect arithmetic, product confusion, refusal behavior, and known scoring issues.

Improved factual accuracy can coexist with worse refusal behavior. Read both columns. Lower prompt-token counts show less input to the answering model, not total end-to-end speed: embedding and reranking also cost time. Local BGE has no provider request charge, but hardware/electricity remain real costs. API prices are not assumed here. Only when both YAML price fields are filled do we estimate token charges from provider-reported usage (including separately reported Gemini thought tokens). Qwen estimates never substitute for billable usage. Missing prices or usage show unavailable; known usage totals are not a complete bill and exclude discounts, taxes, and other charges. Three-question smoke results only verify plumbing; they do not support a model ranking.

Each `12_rag_<model>_<condition>.json` records answers/statuses, selected chunks and scores, prompt tokens, exact protocol, index identity, YAML and effective-configuration hashes, and the matched comparison. On changed identities, the writer preserves the prior file and adds a hash suffix. Saved notebook outputs remain empty until you run it.

In [ ]:
# Save reproducible per-condition results, including scores, raw answers, context and experiment identity.
# Different experiment identities keep the old filename and give the new result a hash suffix; the original lab runs stay intact.
# A nested dictionary/list mirrors JSON structure. Lists of request hashes identify exactly which answers were measured.

OUTPUT_PATHS = []
if RUN:
    for model_id in SELECTED:
        # Each condition changes the evidence block; the selected model and its generation settings remain fixed.
        for condition in CONDITIONS:
            rows = [r for r in records if r["model_id"] == model_id and r["condition"] == condition]
            expected = expected_items(condition)
            candidate = READY.get(model_id)
            identity = digest({"requests": [r["cache_key"] for r in rows],
                               "model": model_id, "condition": condition,
                               "questions": expected, "retrieval": RETRIEVAL_IDENTITY,
                               "inventory": next(r for r in INVENTORY if r["model"] == model_id)})
            summary = next(r for r in summary_rows if r["model"] == model_id and r["condition"] == condition)
            payload = {"model": model_id, "condition": condition, "split": "dev", "answers": rows,
                       "summary": summary, "protocol": protocol(candidate) if candidate else None,
                       "index_name": RAG["index_name"], "index_content_sha256": INDEX_CONTENT_SHA256,
                       "retrieval_identity": RETRIEVAL_IDENTITY, "retrieval_snapshot": str(RETRIEVAL_PATH),
                       "rag_yaml_sha256": RAG_YAML_SHA256, "effective_config_sha256": EFFECTIVE_CONFIG_SHA256,
                       "config": REDACTED_RAG, "versions": VERSIONS, "run_notes": run_notes,
                       "skipped_conditions": SKIPPED_CONDITIONS, "gold_labels_available": HAS_GOLD_LABELS,
                       "effective_search_settings": SEARCH_CFG["retrieval"],
                       "inventory": [r for r in INVENTORY if r["model"] == model_id],
                       "delta_percentage_points": [r for r in delta_rows if r["model"] == model_id and r["condition"] == condition],
                       "api_attempts_this_execution": api_calls_this_execution,
                       "notes": "Only completed nonempty responses scored; historical unmatched runs excluded from deltas; heuristic scorer limits apply."}
            path = write_run_artifact(PREFIX + "_" + model_id + "_" + condition, payload, identity)
            OUTPUT_PATHS.append(path)
    print("Saved", len(OUTPUT_PATHS), "condition files plus the retrieval snapshot.")
    for path in OUTPUT_PATHS:
        print(path)
else:
    print("Preview complete. Edit configs/rag.yaml to enable the independent experiment.")

## Save one result file per model

The RAG config names `cpt`, `sft`, and `cpt_sft_merged`. The third entry is the full base + CPT + SFT additive merge built above. Its saved protocol includes the source hashes and merge formula. It is distinct from the sequential CPT→SFT training example in notebook 10. This saving cell only bundles the completed evaluation; it does not alter model weights.

The three names below contain **reranked RAG answers on dev** as their main `answers` and `summary`, plus all available conditions for comparison. Missing checkpoints produce an explicit `SKIPPED` report with their path and no invented scores. The standard per-condition files above remain available. If a named file already contains a different experiment, its previous bytes are archived before replacing that named latest result.

- `12_RAG+CPT.json`
- `13_RAG+SFT.json`
- `14_RAG+CPT+SFT.json`


In [ ]:
# Read the model-to-filename mapping for the named result files.
# dict.get uses the built-in mapping if the YAML does not provide one. This mapping is data; no file is written yet.
# This is part 1 of the original cell; run the parts in order.

# Purpose: collect each adapter's RAG result into a fixed filename.
# Why: you can open one model-level JSON instead of finding four condition files.
# How: keep rag_rerank as the main result and retain the other conditions underneath.
# Syntax: dict.get supplies a default; comprehensions filter existing results by model.
MODEL_RUN_NAMES = EV.get("named_model_runs", {
    "cpt": "12_RAG+CPT", "sft": "13_RAG+SFT", "cpt_sft_merged": "14_RAG+CPT+SFT"
})

In [ ]:
# Save a named latest result while archiving earlier different results.
# The regular expression validates the whole safe filename; SHA-256 names the archive by its previous bytes. A temporary file protects against partial writes.
# def defines this reusable operation; its body runs only when the function is called later.


def write_named_model_run(stem, payload):
    # re.fullmatch checks the whole filename and excludes directory separators.
    if not re.fullmatch(r"[0-9]{2}_RAG(?:\+[A-Z0-9_]+)+", stem):
        raise ValueError("Named run must use a safe name such as 12_RAG+CPT")
    path = lab.RUNS / (stem + ".json")
    if path.exists():
        previous = path.read_bytes()
        if json.loads(previous).get("identity") != payload["identity"]:
            # Preserve an earlier experiment before updating this easy-to-find name.
            archive = lab.RUNS / "rag_archive"
            archive.mkdir(exist_ok=True)
            archive_path = archive / (stem + "_" + hashlib.sha256(previous).hexdigest()[:16] + ".json")
            if archive_path.exists() and archive_path.read_bytes() != previous:
                raise ValueError("Archive hash collision; the previous result was not overwritten")
            archive_path.write_bytes(previous)
    # Write to a temporary file, then replace atomically: interrupted writes do not
    # leave half a JSON file at the requested result path.
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(path)
    return path

In [ ]:
# Collect and save the selected adapters' named result bundles when execution is enabled.
# The main answers are reranked RAG; all other conditions are retained alongside them. A missing checkpoint yields SKIPPED with no answers.
# This is part 3 of the original cell; run the parts in order.


# Preview mode creates no run files. Only selected, configured named models are saved.
NAMED_OUTPUT_PATHS = []
if RUN:
    # .items() yields (key, value) pairs: model registry ID and its output filename.
    for model_id, stem in MODEL_RUN_NAMES.items():
        if model_id not in SELECTED:
            continue
        model_inventory = next(row for row in INVENTORY if row["model"] == model_id)
        by_condition = {}
        # Each condition changes the evidence block; the selected model and its generation settings remain fixed.
        for condition in CONDITIONS:
            by_condition[condition] = {
                "summary": next(row for row in summary_rows if row["model"] == model_id and row["condition"] == condition),
                "answers": [row for row in records if row["model_id"] == model_id and row["condition"] == condition],
                "delta_percentage_points": [row for row in delta_rows if row["model"] == model_id and row["condition"] == condition],
            }
        primary = by_condition.get("rag_rerank", {"summary": None, "answers": []})
        # Missing artifacts get a truthful status, not base-model answers or zero scores.
        complete = bool(primary["summary"] and primary["summary"]["state"] == "complete")
        state = "SKIPPED" if model_id not in READY else "complete" if complete else "incomplete"
        payload = {
            "name": stem, "model": model_id, "status": state, "split": "dev",
            "primary_condition": "rag_rerank", "summary": primary["summary"],
            "answers": primary["answers"], "conditions": by_condition,
            "inventory": model_inventory, "requested_question_count": len(items),
            "index_name": RAG["index_name"], "retrieval_snapshot": str(RETRIEVAL_PATH),
            "rag_yaml_sha256": RAG_YAML_SHA256, "effective_config_sha256": EFFECTIVE_CONFIG_SHA256,
            "protocol": protocol(READY[model_id]) if model_id in READY else None,
            "saved_at": datetime.now(timezone.utc).isoformat(),
            "identity": digest({"model": model_id, "inventory": model_inventory,
                                "requests": [row["cache_key"] for row in records if row["model_id"] == model_id],
                                "retrieval": RETRIEVAL_IDENTITY, "primary_condition": "rag_rerank"}),
        }
        NAMED_OUTPUT_PATHS.append(write_named_model_run(stem, payload))
    for path in NAMED_OUTPUT_PATHS:
        print("Named result:", path)
else:
    print("Named files will be saved when evaluation.run is true:", MODEL_RUN_NAMES)